# Exercise 7.1: Energy loss in a scintillator tile

From *Programming in High Energy Particle Physics*, Chapter 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch07/ex07_1_solution.ipynb)

Exercise 7.1 Energy loss in a scintillator tile Build a simple Geant4 simulation of a single plastic scintillator tile (5 cm × 5 cm × 1 cm). Fire 1 GeV muons at it perpendicular to the large face and measure the energy deposit distribution. Muons at this energy are close to minimum ionizing and give a clean energy-loss signal. Compare the result with the Landau–Vavilov prediction, and discuss why the Bichsel description of straggling in thin layers fits better. Repeat with 100 MeV electrons and explain why their distribution is much broader and less well described by energy-loss straggling alone.

<details><summary>Hint</summary>

Use G4_PLASTIC_SC_VINYLTOLUENE or G4_POLYSTYRENE from the NIST database, fill one histogram per event with the total deposit in the tile, and compare the most probable value (MPV) and the width with the formula for \(\Delta_p\) in the PDG review "Passage of particles through matter". Electrons radiate bremsstrahlung and scatter strongly even in 1 cm of plastic.

</details>

### Colab setup and local Geant4 build

The Colab path below is a **lightweight stand-in**, not Geant4 transport. The optional `geant4_pybind` wheel is large and can download large physics datasets. A full C++ Geant4 build needs Geant4 installed locally; save the generated `.cc` and `CMakeLists.txt`, then run the shown CMake commands locally. The C++ program is not executed by Colab.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy scipy matplotlib
INSTALL_GEANT4_BINDINGS = False  # optional, heavy; not needed for the stand-in
if IN_COLAB and INSTALL_GEANT4_BINDINGS:
    %pip install -q geant4_pybind
import numpy as np
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)

### Complete local Geant4 transport program

The tile is 5 cm × 5 cm × 1 cm polystyrene. Each event records the total deposit from the primary and all secondaries in the tile. Use the command line particle choice to repeat for muons and electrons.

In [ ]:
%%writefile ex07_1.cc
#include "G4RunManager.hh"
#include "G4VUserDetectorConstruction.hh"
#include "G4VUserPrimaryGeneratorAction.hh"
#include "G4VUserActionInitialization.hh"
#include "G4UserEventAction.hh"
#include "G4UserSteppingAction.hh"
#include "G4NistManager.hh"
#include "G4Box.hh"
#include "G4LogicalVolume.hh"
#include "G4PVPlacement.hh"
#include "G4ParticleGun.hh"
#include "G4ParticleTable.hh"
#include "G4SystemOfUnits.hh"
#include "G4Event.hh"
#include "G4Step.hh"
#include "FTFP_BERT.hh"
#include "Randomize.hh"
#include <fstream>
#include <string>
#include <cstdlib>
class TileDetector : public G4VUserDetectorConstruction {
public:
  G4LogicalVolume* tile = nullptr;
  G4VPhysicalVolume* Construct() override {
    auto* nist=G4NistManager::Instance();
    auto* air=nist->FindOrBuildMaterial("G4_AIR");
    auto* plastic=nist->FindOrBuildMaterial("G4_POLYSTYRENE");
    auto* world=new G4LogicalVolume(new G4Box("world",10*cm,10*cm,10*cm),air,"world");
    auto* pv=new G4PVPlacement(nullptr,{},world,"world",nullptr,false,0);
    tile=new G4LogicalVolume(new G4Box("tile",2.5*cm,2.5*cm,0.5*cm),plastic,"tile");
    new G4PVPlacement(nullptr,{},tile,"tile",world,false,0);
    return pv;
  }
};
class Beam : public G4VUserPrimaryGeneratorAction {
  G4ParticleGun gun{1};
public:
  explicit Beam(bool electron) {
    gun.SetParticleDefinition(G4ParticleTable::GetParticleTable()->FindParticle(electron?"e-":"mu-"));
    gun.SetParticleEnergy(electron?100*MeV:1*GeV);
    gun.SetParticlePosition(G4ThreeVector(0,0,-2*cm));
    gun.SetParticleMomentumDirection(G4ThreeVector(0,0,1));
  }
  void GeneratePrimaries(G4Event* e) override { gun.GeneratePrimaryVertex(e); }
};
class DepositEvent : public G4UserEventAction {
  std::ofstream out;
  G4double edep=0;
public:
  explicit DepositEvent(const std::string& path):out(path) {out << "edep_MeV\n";}
  void BeginOfEventAction(const G4Event*) override {edep=0;}
  void Add(G4double e) {edep+=e;}
  void EndOfEventAction(const G4Event*) override {out << edep/MeV << '\n';}
};
class DepositStep : public G4UserSteppingAction {
  const TileDetector* det;
  DepositEvent* evt;
public:
  DepositStep(const TileDetector* d,DepositEvent* e):det(d),evt(e){}
  void UserSteppingAction(const G4Step* step) override {
    if(step->GetPreStepPoint()->GetTouchableHandle()->GetVolume()->GetLogicalVolume()==det->tile)
      evt->Add(step->GetTotalEnergyDeposit());
  }
};
class Actions : public G4VUserActionInitialization {
  const TileDetector* det;
  bool electron;
  std::string path;
public:
  Actions(const TileDetector* d,bool e,std::string p):det(d),electron(e),path(p){}
  void Build() const override {
    SetUserAction(new Beam(electron));
    auto* evt=new DepositEvent(path);
    SetUserAction(evt);
    SetUserAction(new DepositStep(det,evt));
  }
};
int main(int argc,char** argv) {
  if(argc<4) return 2; // particle (mu/e), event count, CSV path
  G4Random::setTheSeed(42);
  auto* rm=new G4RunManager;
  auto* det=new TileDetector;
  rm->SetUserInitialization(det);
  rm->SetUserInitialization(new FTFP_BERT);
  rm->SetUserInitialization(new Actions(det,std::string(argv[1])=="e",argv[3]));
  rm->Initialize();
  rm->BeamOn(std::atoi(argv[2]));
  delete rm;
}


In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ex07_1 LANGUAGES CXX)
find_package(Geant4 REQUIRED)
include(${Geant4_USE_FILE})
add_executable(ex07_1 ex07_1.cc)
target_link_libraries(ex07_1 ${Geant4_LIBRARIES})


Local shell commands after extracting the files:

```bash
cmake -S . -B build
cmake --build build -j2
./build/ex07_1 mu 2000 mu.csv
./build/ex07_1 e 2000 electron.csv
```

Set `G4LEDATA` and other Geant4 data variables as required by your installation. These runs use FTFP_BERT and a fixed random seed.
If CMake cannot locate an optional dependency in an existing Geant4 installation, a direct build is also possible with `c++ ex07_N.cc -o ex07_N $(geant4-config --cflags --libs)` (replace `N` with this exercise number and configure the library path for that installation).

### PDG most-probable energy loss

Compute Eq. 34.12 of the [PDG passage review](https://pdg.lbl.gov/2025/reviews/rpp2025-rev-passage-particles-matter.pdf) for a 1 GeV **kinetic-energy** muon. Polystyrene density, mean excitation energy, ⟨Z/A⟩, and Sternheimer density-effect constants come from the [PDG material table](https://pdg.lbl.gov/2024/AtomicNuclearProperties/MUE/muE_polystyrene.pdf). The formula predicts the most probable loss, not the mean deposited energy in a finite tile.

In [ ]:
K=0.307075 # MeV cm²/g
z_over_a=0.53768 # mol/g
rho=1.060 # g/cm³
I=68.7e-6 # MeV
m_mu=105.658 # MeV
m_e=0.510999 # MeV
T_mu=1000.0 # MeV kinetic energy
gamma=1+T_mu/m_mu
beta2=1-gamma**-2
beta_gamma=np.sqrt(gamma**2-1)
x=np.log10(beta_gamma)
a, m_s, x0, x1, Cbar=0.16454,3.2224,0.1647,2.5031,3.2999
if x<x0:
    delta=0.0
elif x<x1:
    delta=2*np.log(10)*x-Cbar+a*(x1-x)**m_s
else:
    delta=2*np.log(10)*x-Cbar
xi=(K/2)*z_over_a*(rho*1.0)/beta2 # 1 cm path, MeV
pdg_mpv=xi*(np.log(2*m_e*beta2*gamma**2/I)+np.log(xi/I)+0.200-beta2-delta)
print(f"PDG ξ={xi:.3f} MeV, δ={delta:.3f}, MPV={pdg_mpv:.2f} MeV")
assert 1.5<pdg_mpv<1.8

### Colab stand-in: illustrative energy-loss shapes

**This is not a Geant4 or Bichsel calculation.** A Moyal density is used as a convenient Landau-like shape, calibrated to the book’s muon MPV and mean. The broad electron sample is only a qualitative illustration.

In [ ]:
from scipy.stats import moyal
n_events=5000
mu_edep=moyal.rvs(loc=1.6,scale=0.30,size=n_events,random_state=rng)
mu_edep=np.clip(mu_edep,0,None)
e_edep=np.clip(rng.gamma(shape=2.0,scale=1.5,size=n_events)+rng.exponential(3.0,size=n_events)*(rng.random(n_events)<0.12),0,None)
fig,ax=plt.subplots()
ax.hist(mu_edep,bins=np.linspace(0,12,80),density=True,histtype="step",label="1 GeV μ (Moyal stand-in)")
ax.hist(e_edep,bins=np.linspace(0,12,80),density=True,histtype="step",label="100 MeV e (toy stand-in)")
ax.set(xlabel="Tile energy deposit [MeV]",ylabel="Density / bin");ax.legend();plt.show()
print(f"Muon stand-in: MPV input 1.6 MeV; sample mean {mu_edep.mean():.2f} MeV")
assert 1.8<mu_edep.mean()<2.2

The book quotes a Landau–Vavilov MPV of **about 1.6 MeV** and a mean loss of **about 2 MeV** for 1 GeV muons in 1 cm of polystyrene. The long upper tail comes from energetic δ electrons; escaping δ rays can lower Geant4’s tile mean. A thin-layer Bichsel calculation models discrete collision straggling more faithfully than a limiting Landau shape. Electrons also radiate bremsstrahlung and scatter strongly, so a pure ionization-straggling model misses their broad response. The stand-in cannot validate these transport effects. In 1 cm of plastic (a small fraction of a radiation length), bremsstrahlung may be rare; an actual Geant4 electron histogram need not be dramatically broader in a small sample. Quantify the difference from the transport CSV instead of assuming it.

### Analyze real Geant4 CSVs when available

In [ ]:
from pathlib import Path
real={name:Path(name+".csv") for name in ("mu","electron")}
if all(p.exists() for p in real.values()):
    fig,ax=plt.subplots()
    for name,p in real.items():
        x=np.genfromtxt(p,delimiter=",",names=True)["edep_MeV"]
        counts,edges=np.histogram(x,bins=np.linspace(0,10,101))
        mpv=0.5*(edges[np.argmax(counts)]+edges[np.argmax(counts)+1])
        print(f"{name}: N={len(x)}, histogram MPV≈{mpv:.2f} MeV, mean={x.mean():.2f} MeV, SD={x.std(ddof=1):.2f} MeV")
        ax.hist(x,bins=np.linspace(0,10,101),density=True,histtype="step",label=name)
    ax.set(xlabel="Tile energy deposit [MeV]",ylabel="Density / bin");ax.legend();plt.show()
else:
    print("Run the local Geant4 commands and place mu.csv and electron.csv beside this notebook to compare transport results.")